# Blackjack baseline: dynamic episodic TOMC + tuned KL-SPMD

**IE 692 course project.** Simplified Blackjack (cards 2-5 drawn with replacement,
dealer draws to 9 or more, bust above 11). Each iteration follows:

**Freeze policy -> play hands until every eligible pair has L(s,a) >= 100 suffix steps ->
average every complete discounted return into Q -> one KL-SPMD update.**

What is run:
1. **Checks.** Estimator arithmetic, the KL update, and the exact model.
2. **Tuning.** Stepsize eta in {0.1, 1, 2, 10}, each with 5% of the budget (150,000 observations).
3. **Full runs.** The selected eta on five seeds, 3,000,000 training observations each.
4. **Evaluation (never used for training).** Exact Bellman V^pi at the reference state
   (player sum 8, dealer card 4) after every update, the optimal V* by exact dynamic
   programming, and the loss rate of every policy over 500 separate test games.
5. **Plots.** Value, loss rate, and completed policy updates against cumulative training
   observations: mean of five seeds, staircase curves, shaded 95% confidence intervals.

**How to run:** `python blackjack_baseline.py` (or run the `# %%` cells in VS Code / Jupyter).
Dependencies: NumPy, pandas, Matplotlib. Figures go to `figures/`, curves to `results/`.

In [ ]:
# 1. Settings
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MaxNLocator

GAMMA = 0.9
MIN_SUFFIX_STEPS = 100                    # Batch ends once every eligible pair has L(s,a) >= 100.
SAMPLE_BUDGET = 400_000                 # Training observations (player state-action transitions) per run.
TUNING_BUDGET = SAMPLE_BUDGET // 20       # 5% of the budget = 150,000 observations per candidate.
ETA_CANDIDATES = (0.1, 1.0, 2.0, 10.0)    # Stepsizes compared in the tuning step.
SEEDS = (0, 1, 2, 3, 4)                   # Five independent full runs.
TUNING_SEED = 100                         # Tuning uses its own seed, separate from the full runs.
TEST_GAMES = 500                          # Independent loss-rate evaluation games per policy.

CARD_VALUES = (2, 3, 4, 5)                # Infinite deck, drawn with replacement.
BUST_LIMIT = 11                           # A sum above 11 is a bust.
DEALER_STANDS_AT = 9                      # Dealer draws until the sum is 9 or more.
WIN, DRAW, LOSS = -1.0, 0.0, 1.0          # Costs to minimize.

PLAYER_SUMS = range(4, 12)                # 8 player sums (4..11)
DEALER_CARDS = range(2, 6)                # 4 dealer up-cards (2..5)
N_STATES, N_ACTIONS = 32, 2               # All 32 states kept; none excluded as unreachable.
STICK, HIT = 0, 1
ACTION_NAMES = ("stick", "hit")
PI_THRESHOLD = (1 - GAMMA) / N_ACTIONS    # 0.05: pairs below it get the pessimistic fallback.
FALLBACK_Q = 1 / (1 - GAMMA)              # 10.0


def state2ID(player_sum, dealer_card):
    """Map (player sum 4..11, dealer card 2..5) to a state index 0..31."""
    return (player_sum - 4) * 4 + (dealer_card - 2)


REF_PLAYER_SUM, REF_DEALER_CARD = 8, 4
REPORT_STATE = state2ID(REF_PLAYER_SUM, REF_DEALER_CARD)
REPORT_LABEL = f"({REF_PLAYER_SUM}, {REF_DEALER_CARD})"

# Plot style shared with the GridWorld notebook (same palette, fonts and layout).
BASELINE_COLOR = "#6f6e69"
INK = "#0b0b0b"
MUTED = "#52514e"
BACKGROUND = "#fcfcfb"
GRID_COLOR = "#e6e5e0"
FIGURES, RESULTS = Path("figures"), Path("results")

## 2. Environment: play one hand under a frozen policy

The player starts with two cards and sees one dealer card. HIT draws a card: cost 1 on a
bust (the hand ends), otherwise cost 0. STICK ends the hand: the dealer draws to 9+ and the
cost is -1 (win), 0 (draw) or +1 (loss). A hand returns its states, actions and costs.
Random draws are buffered for speed; the sequence is still fully set by the seed.

In [ ]:
class RandomStream:
    """Buffered cards and uniforms from one seeded NumPy generator."""

    def __init__(self, rng, block=1 << 16):
        self.rng, self.block = rng, block
        self._cards, self._uniforms = [], []

    def card(self):
        if not self._cards:
            self._cards = self.rng.choice(CARD_VALUES, size=self.block).tolist()
        return self._cards.pop()

    def uniform(self):
        if not self._uniforms:
            self._uniforms = self.rng.random(self.block).tolist()
        return self._uniforms.pop()


def stick_cost(player_sum, dealer_card, stream):
    """Dealer draws to DEALER_STANDS_AT or more; return the player's terminal cost."""
    dealer_sum = dealer_card
    while dealer_sum < DEALER_STANDS_AT:
        dealer_sum += stream.card()
    if dealer_sum > BUST_LIMIT or player_sum > dealer_sum:
        return WIN
    return LOSS if player_sum < dealer_sum else DRAW


def play_hand(hit_prob, stream):
    """One complete hand. hit_prob[s] = pi(HIT | s). Returns (states, actions, costs)."""
    player_sum = stream.card() + stream.card()
    dealer_card = stream.card()
    states, actions, costs = [], [], []
    while True:
        state = state2ID(player_sum, dealer_card)
        states.append(state)
        if stream.uniform() < hit_prob[state]:
            actions.append(HIT)
            player_sum += stream.card()
            if player_sum > BUST_LIMIT:
                costs.append(LOSS)
                return states, actions, costs
            costs.append(0.0)
        else:
            actions.append(STICK)
            costs.append(stick_cost(player_sum, dealer_card, stream))
            return states, actions, costs

## 3. Episodic batch collection and Monte Carlo Q

Eligible pairs: $\mathcal E_k=\{(s,a):\pi_k(a\mid s)\ge\underline\pi\}$, $\underline\pi=0.05$.
For an occurrence of $(s,a)$ at step $t$ of a hand of length $T$, add $T-t$ to the suffix-step
accumulator $L(s,a)$, add its complete discounted return $G_t=\sum_{j\ge t}\gamma^{j-t}C_j$ to a
running sum, and add 1 to the return count $N(s,a)$. Returns never cross hand boundaries.
The batch ends after the first hand at which **every** eligible pair has $L(s,a)\ge100$;
frequent pairs keep contributing until then. Then

$$\widehat Q_k(s,a)=\frac{1}{N(s,a)}\sum_{i=1}^{N(s,a)}G^{(i)}(s,a)\ \text{ if }(s,a)\in\mathcal E_k,
\qquad \widehat Q_k(s,a)=\frac{1}{1-\gamma}\ \text{ otherwise.}$$

If the observation budget runs out first, the current hand is finished and the incomplete
batch is not used for an update.

In [ ]:
def empty_batch():
    pairs = N_STATES * N_ACTIONS
    return {"return_sum": [0.0] * pairs, "return_count": [0] * pairs, "suffix_steps": [0] * pairs}


def add_hand(batch, states, actions, costs, gamma, eligible, min_steps):
    """Add every complete return of one hand; return how many eligible pairs just reached min_steps."""
    hand_length, G, newly_complete = len(costs), 0.0, 0
    for t in range(hand_length - 1, -1, -1):
        G = costs[t] + gamma * G
        pair = states[t] * N_ACTIONS + actions[t]
        before = batch["suffix_steps"][pair]
        batch["suffix_steps"][pair] += hand_length - t
        batch["return_sum"][pair] += G
        batch["return_count"][pair] += 1
        if eligible[pair] and before < min_steps <= batch["suffix_steps"][pair]:
            newly_complete += 1
    return newly_complete


def collect_batch(policy, stream, gamma=GAMMA, min_steps=MIN_SUFFIX_STEPS,
                  observations_left=np.inf):
    """Play hands under the frozen policy until all eligible pairs have L >= min_steps."""
    eligible_mask = policy >= PI_THRESHOLD
    eligible = eligible_mask.ravel().tolist()
    hit_prob = policy[:, HIT].tolist()
    batch, missing, observations, hands = empty_batch(), int(eligible_mask.sum()), 0, 0

    while missing > 0 and observations < observations_left:
        states, actions, costs = play_hand(hit_prob, stream)
        missing -= add_hand(batch, states, actions, costs, gamma, eligible, min_steps)
        observations += len(costs)
        hands += 1

    counts = np.array(batch["return_count"]).reshape(policy.shape)
    steps = np.array(batch["suffix_steps"]).reshape(policy.shape)
    sums = np.array(batch["return_sum"]).reshape(policy.shape)
    q_hat = np.full(policy.shape, FALLBACK_Q)
    q_hat[eligible_mask] = sums[eligible_mask] / np.maximum(counts[eligible_mask], 1)
    return {"q_hat": q_hat, "complete": missing == 0, "observations": observations,
            "hands": hands, "return_count": counts, "suffix_steps": steps,
            "eligible": eligible_mask}

## 4. KL-SPMD policy update

$$\pi_{k+1}(a\mid s)=\frac{\pi_k(a\mid s)\exp[-\eta\widehat Q_k(s,a)]}
{\sum_b\pi_k(b\mid s)\exp[-\eta\widehat Q_k(s,b)]}$$

Costs are minimized, hence the minus sign. Subtracting each row's largest exponent keeps
`exp` stable without changing the result.

In [ ]:
def kl_spmd_update(policy, q_hat, eta):
    scores = -eta * q_hat
    scores -= scores.max(axis=1, keepdims=True)
    weights = policy * np.exp(scores)
    return weights / weights.sum(axis=1, keepdims=True)

## 5. Exact model, used only for evaluation

The dealer's final-sum distribution is computed exactly for each up-card, which gives the
expected STICK cost. HIT moves to $(p+c, d)$ with probability 1/4 per card, or busts (cost 1).
Then $V^{\pi}=(I-\gamma P_\pi)^{-1}c_\pi$ and
$V^*(s)=\min_a[c(s,a)+\gamma\sum_{s'}P(s'\mid s,a)V^*(s')]$.
The hand terminates with value 0, so $P$ is substochastic.

In [ ]:
def dealer_final_distribution(dealer_card):
    """Exact probabilities of the dealer's final sum, starting from the up-card."""
    final, frontier = {}, {dealer_card: 1.0}
    while frontier:
        next_frontier = {}
        for total, prob in frontier.items():
            for card in CARD_VALUES:
                new_total, new_prob = total + card, prob / len(CARD_VALUES)
                target = final if new_total >= DEALER_STANDS_AT else next_frontier
                target[new_total] = target.get(new_total, 0.0) + new_prob
        frontier = next_frontier
    return final


def build_exact_model():
    """Return P[state, action, next_state] (substochastic) and c[state, action]."""
    P = np.zeros((N_STATES, N_ACTIONS, N_STATES))
    c = np.zeros((N_STATES, N_ACTIONS))
    for dealer_card in DEALER_CARDS:
        dealer = dealer_final_distribution(dealer_card)
        for player_sum in PLAYER_SUMS:
            s = state2ID(player_sum, dealer_card)
            for dealer_sum, prob in dealer.items():
                if dealer_sum > BUST_LIMIT or player_sum > dealer_sum:
                    c[s, STICK] += prob * WIN
                elif player_sum < dealer_sum:
                    c[s, STICK] += prob * LOSS
            for card in CARD_VALUES:
                prob = 1 / len(CARD_VALUES)
                if player_sum + card > BUST_LIMIT:
                    c[s, HIT] += prob * LOSS
                else:
                    P[s, HIT, state2ID(player_sum + card, dealer_card)] += prob
    return P, c


def exact_values(policy, P, c, gamma=GAMMA):
    """Exact V^pi for all 32 states (Bellman linear system)."""
    P_pi = np.einsum("sa,sat->st", policy, P)
    c_pi = np.sum(policy * c, axis=1)
    return np.linalg.solve(np.eye(N_STATES) - gamma * P_pi, c_pi)


def optimal_values(P, c, gamma=GAMMA, tol=1e-13):
    """Exact V* by value iteration (the player sum only increases, so it converges in a few sweeps)."""
    V = np.zeros(N_STATES)
    while True:
        V_new = np.min(c + gamma * P @ V, axis=1)
        if np.max(np.abs(V_new - V)) < tol:
            return V_new
        V = V_new


P_exact, c_exact = build_exact_model()
V_STAR = optimal_values(P_exact, c_exact)
UNIFORM_POLICY = np.full((N_STATES, N_ACTIONS), 1 / N_ACTIONS)
V_UNIFORM = exact_values(UNIFORM_POLICY, P_exact, c_exact)
OPTIMAL_ACTIONS = np.argmin(c_exact + GAMMA * P_exact @ V_STAR, axis=1)

print(f"Uniform policy: V{REPORT_LABEL} = {V_UNIFORM[REPORT_STATE]:.4f}")
print(f"Optimal policy: V*{REPORT_LABEL} = {V_STAR[REPORT_STATE]:.4f}")
print(f"Largest possible improvement at {REPORT_LABEL}: "
      f"{V_UNIFORM[REPORT_STATE] - V_STAR[REPORT_STATE]:.4f}")
print("Optimal action (rows: player sum 4-11, columns: dealer card 2-5; S = stick, H = hit):")
for player_sum in PLAYER_SUMS:
    row = ["SH"[OPTIMAL_ACTIONS[state2ID(player_sum, d)]] for d in DEALER_CARDS]
    print(f"  {player_sum:2d}: {' '.join(row)}")

## 6. Small checks before running

Scripted hands test the return arithmetic, the separate N and L counters, and the
fallback. The KL update is checked against the closed-form sigmoid of the original
script (constant Q), the direct formula, and invariance to per-state shifts. The exact
model is checked against simulation.

In [ ]:
def check_state_ids():
    ids = [state2ID(p, d) for p in PLAYER_SUMS for d in DEALER_CARDS]
    assert sorted(ids) == list(range(N_STATES))
    print("PASS: 32 distinct state IDs.")


def check_episodic_estimator():
    gamma = 0.5
    eligible = [True] * (N_STATES * N_ACTIONS)
    batch = empty_batch()
    # Hand 1: hit at state 0, hit at state 5, stick at state 9 with a loss.
    newly = add_hand(batch, [0, 5, 9], [HIT, HIT, STICK], [0.0, 0.0, 1.0], gamma, eligible, 3)
    assert newly == 1                                  # Only (0, HIT) has L = 3 so far.
    # Hand 2: (0, HIT) again, then a bust. Returns never mix across hands.
    add_hand(batch, [0], [HIT], [1.0], gamma, eligible, 3)
    pair = 0 * N_ACTIONS + HIT
    assert batch["return_count"][pair] == 2 and batch["suffix_steps"][pair] == 3 + 1
    np.testing.assert_allclose(batch["return_sum"][pair], gamma**2 * 1.0 + 1.0)
    np.testing.assert_allclose(batch["return_sum"][5 * N_ACTIONS + HIT], gamma * 1.0)
    assert batch["suffix_steps"][9 * N_ACTIONS + STICK] == 1

    # A full batch: every eligible pair reaches L >= 100, ineligible pairs get the fallback.
    policy = UNIFORM_POLICY.copy()
    policy[REPORT_STATE] = [1 - PI_THRESHOLD / 2, PI_THRESHOLD / 2]
    result = collect_batch(policy, RandomStream(np.random.default_rng(7)))
    assert result["complete"]
    assert (result["suffix_steps"][result["eligible"]] >= MIN_SUFFIX_STEPS).all()
    assert result["observations"] == result["return_count"].sum()
    assert result["q_hat"][REPORT_STATE, HIT] == FALLBACK_Q
    eligible_q = result["q_hat"][result["eligible"]]
    assert (eligible_q >= -1).all() and (eligible_q <= 1).all()

    # Budget stop: the current hand is finished, and the batch is marked incomplete.
    short = collect_batch(UNIFORM_POLICY, RandomStream(np.random.default_rng(8)), observations_left=10)
    assert not short["complete"] and 10 <= short["observations"] <= 10 + N_STATES
    print("PASS: complete returns, separate N and L, no cross-hand returns, fallback, budget stop.")


def check_kl_update():
    # Original verification: constant Q (stick 0, hit 1) gives P(stick)_k = 1 / (1 + exp(-eta k)).
    fake_q = np.tile([0.0, 1.0], (N_STATES, 1))
    policy = UNIFORM_POLICY.copy()
    for k in range(1, 51):
        policy = kl_spmd_update(policy, fake_q, 0.1)
        np.testing.assert_allclose(policy[:, STICK], 1 / (1 + np.exp(-0.1 * k)), rtol=1e-12)

    rng = np.random.default_rng(2026)
    pi = rng.dirichlet(np.ones(N_ACTIONS), size=N_STATES)
    pi[0] = [1.0, 0.0]                                   # Deterministic row stays deterministic.
    q = rng.uniform(-1, 1, size=pi.shape)
    for eta in (0.0, 0.1, 1.0, 10.0):
        actual = kl_spmd_update(pi, q, eta)
        expected = pi * np.exp(-eta * q)
        expected /= expected.sum(axis=1, keepdims=True)
        np.testing.assert_allclose(actual, expected, rtol=1e-12, atol=1e-12)
        assert np.all(np.sum(actual * q, axis=1) <= np.sum(pi * q, axis=1) + 1e-12)
        shifted = kl_spmd_update(pi, q + rng.uniform(-100, 100, size=(N_STATES, 1)), eta)
        np.testing.assert_allclose(shifted, actual, rtol=1e-10, atol=1e-12)
    print("PASS: closed-form sigmoid, direct formula, fixed-Q improvement, row shifts.")


def check_exact_model(n_hands=200_000, seed=2028):
    """Monte Carlo Q of the uniform policy should match the exact Q^pi within 4 standard errors."""
    assert np.isclose(sum(dealer_final_distribution(4).values()), 1.0)
    V = exact_values(UNIFORM_POLICY, P_exact, c_exact)
    Q_exact = c_exact + GAMMA * P_exact @ V
    stream = RandomStream(np.random.default_rng(seed))
    hit_prob = UNIFORM_POLICY[:, HIT].tolist()
    returns = [[[] for _ in range(N_ACTIONS)] for _ in range(N_STATES)]
    for _ in range(n_hands):
        states, actions, costs = play_hand(hit_prob, stream)
        G = 0.0
        for t in range(len(costs) - 1, -1, -1):
            G = costs[t] + GAMMA * G
            returns[states[t]][actions[t]].append(G)
    worst = 0.0
    for s in range(N_STATES):
        for a in range(N_ACTIONS):
            sample = np.array(returns[s][a])
            se = sample.std(ddof=1) / np.sqrt(len(sample))
            if se == 0:                                  # e.g. HIT at 11 always busts.
                np.testing.assert_allclose(sample.mean(), Q_exact[s, a])
                continue
            worst = max(worst, abs(sample.mean() - Q_exact[s, a]) / se)
    assert worst < 4.5, worst
    print(f"PASS: exact Q^pi matches simulation (largest deviation {worst:.2f} standard errors).")


check_state_ids()
check_episodic_estimator()
check_kl_update()
check_exact_model()

## 7. The training loop and independent evaluation

Policy $\pi_0$ is uniform. Each completed batch gives one update. A run stops once
`max_observations` training observations are used (the last hand is finished).
After the run, every policy $\pi_0,\dots,\pi_K$ is evaluated twice, outside training:
exact $V^{\pi_k}$, and the loss rate in 500 test games drawn from a separate random stream.

In [ ]:
def run_spmd(max_observations, eta, seed, verbose=True):
    """KL-SPMD + episodic TOMC until the training observation budget is used."""
    train_seed, test_seed = np.random.SeedSequence(seed).spawn(2)
    stream = RandomStream(np.random.default_rng(train_seed))
    policy = UNIFORM_POLICY.copy()
    policies, samples, records = [policy.copy()], [0], []
    used = 0
    started = time.perf_counter()

    while used < max_observations:
        batch = collect_batch(policy, stream, observations_left=max_observations - used)
        used += batch["observations"]
        if not batch["complete"]:
            break                                    # Budget ran out mid-batch: no update.
        q_hat = batch["q_hat"]
        records.append({
            "iteration": len(records),
            "observations": batch["observations"],
            "hands": batch["hands"],
            "cumulative_observations": used,
            "eligible_pairs": int(batch["eligible"].sum()),
            "fallback_pairs": int((~batch["eligible"]).sum()),
            "min_eligible_L": int(batch["suffix_steps"][batch["eligible"]].min()),
            "value_estimate": float(policy[REPORT_STATE] @ q_hat[REPORT_STATE]),
        })
        policy = kl_spmd_update(policy, q_hat, eta)
        policies.append(policy.copy())
        samples.append(used)
        k = len(records)
        if verbose and (k == 1 or k % 25 == 0):
            print(f"eta={eta:g}, seed={seed} | update {k}: {batch['observations']:,} observations "
                  f"({used:,} in total); {batch['eligible'].sum()} eligible pairs", flush=True)

    seconds = time.perf_counter() - started
    if verbose:
        print(f"eta={eta:g}, seed={seed} | done: {len(records)} updates, {used:,} observations, "
              f"{seconds:.0f} s", flush=True)
    return {"seed": seed, "eta": eta, "policies": np.array(policies),
            "samples": np.array(samples), "history": pd.DataFrame(records),
            "total_observations": used, "seconds": seconds, "test_seed": test_seed}


def loss_rate(policy, stream, n_games=TEST_GAMES):
    """Fraction of independent test games that end in a loss (bust or lower than the dealer)."""
    hit_prob = policy[:, HIT].tolist()
    return np.mean([play_hand(hit_prob, stream)[2][-1] == LOSS for _ in range(n_games)])


def summarize_run(result):
    """Evaluate every policy of a run: exact values and independent loss rates."""
    test_stream = RandomStream(np.random.default_rng(result["test_seed"]))
    policies = result["policies"]
    return {
        "seed": result["seed"],
        # samples[k] training observations were used to obtain policy k; policy 0 is uniform.
        "samples": result["samples"],
        "values": np.array([exact_values(pi, P_exact, c_exact) for pi in policies]),
        "loss_rate": np.array([loss_rate(pi, test_stream) for pi in policies]),
        "updates": np.arange(len(policies)),
        "observations": result["history"]["observations"].to_numpy(),
        "seconds": result["seconds"],
        "final_policy": policies[-1],
    }

## 8. Observation-aligned curves and 95% confidence intervals

At any observation count, each seed carries forward its latest completed policy (no
interpolation). Cross-seed means use a Student-t 95% interval with n-1 degrees of freedom.

In [ ]:
T_975 = {2: 12.706, 3: 4.303, 4: 3.182, 5: 2.776, 6: 2.571, 7: 2.447, 8: 2.365, 9: 2.306, 10: 2.262}


def on_grid(run, points, metric):
    """Carry each seed's latest policy forward: the value of `metric` at each observation count."""
    index = np.searchsorted(run["samples"], points, side="right") - 1
    return np.asarray(run[metric])[index]


def mean_ci(curves):
    """Mean and 95% half-width across seeds (rows)."""
    curves = np.asarray(curves, dtype=float)
    n = curves.shape[0]
    half_width = T_975[n] * curves.std(axis=0, ddof=1) / np.sqrt(n)
    return curves.mean(axis=0), half_width


def update_points(runs, budget):
    """Every observation count at which some seed changes policy, plus 0 and the budget."""
    points = np.unique(np.concatenate([[0, budget]] + [run["samples"] for run in runs]))
    return points[points <= budget]


def format_mean_ci(values):
    mean, half_width = mean_ci(np.asarray(values, dtype=float)[:, None])
    return f"{mean[0]:.4f} ± {half_width[0]:.4f}"


def save_runs(name, runs, **settings):
    RESULTS.mkdir(exist_ok=True)
    arrays = {f"seed{run['seed']}_{key}": np.asarray(run[key]) for run in runs
              for key in ("samples", "values", "loss_rate", "updates", "observations", "final_policy")}
    path = RESULTS / f"{name}.npz"
    np.savez(path, seeds=[run["seed"] for run in runs], seconds=[run["seconds"] for run in runs],
             **arrays, **settings)
    return path

## 9. Tune the stepsize with 5% of the budget

Each candidate is run once on the tuning seed with 150,000 observations. The score is the
exact value at the reference state of the last policy obtained within that budget
(smaller cost is better). Tuning observations are not part of the full runs.

In [ ]:
def value_at_reference(run):
    return run["values"][:, REPORT_STATE]


tuning_rows = []
for eta in ETA_CANDIDATES:
    run = summarize_run(run_spmd(TUNING_BUDGET, eta, TUNING_SEED, verbose=False))
    final_values = run["values"][-1]
    tuning_rows.append({
        "eta": eta,
        "policy_updates": int(run["updates"][-1]),
        f"V_at_{REF_PLAYER_SUM}_{REF_DEALER_CARD}": final_values[REPORT_STATE],
        "gap_at_reference": final_values[REPORT_STATE] - V_STAR[REPORT_STATE],
        "mean_gap_all_states": np.mean(final_values - V_STAR),
        "loss_rate": run["loss_rate"][-1],
        "seconds": run["seconds"],
    })

tuning = pd.DataFrame(tuning_rows)
ETA_TUNED = float(tuning.loc[tuning["gap_at_reference"].idxmin(), "eta"])
print(f"Tuning budget: {TUNING_BUDGET:,} observations per candidate (seed {TUNING_SEED}).")
print(tuning.round(4).to_string(index=False))
print(f"\nSelected stepsize: eta = {ETA_TUNED:g}")

## 10. Full runs on five seeds

In [ ]:
def check_run(result):
    history = result["history"]
    assert np.allclose(result["policies"].sum(axis=2), 1) and (result["policies"] >= 0).all()
    assert (history["min_eligible_L"] >= MIN_SUFFIX_STEPS).all()
    np.testing.assert_array_equal(history["cumulative_observations"], history["observations"].cumsum())
    assert SAMPLE_BUDGET <= result["total_observations"] < SAMPLE_BUDGET + N_STATES


results = [run_spmd(SAMPLE_BUDGET, ETA_TUNED, seed) for seed in SEEDS]
for result in results:
    check_run(result)
runs = [summarize_run(result) for result in results]

results_path = save_runs(
    "blackjack_baseline", runs,
    eta=ETA_TUNED, gamma=GAMMA, sample_budget=SAMPLE_BUDGET, min_suffix_steps=MIN_SUFFIX_STEPS,
    report_state=REPORT_STATE, test_games=TEST_GAMES, V_star=V_STAR, V_uniform=V_UNIFORM,
)
print(f"\nSaved {results_path}")

budget = np.array([SAMPLE_BUDGET])
summary = pd.DataFrame([{
    "seed": run["seed"],
    "policy_updates": int(run["updates"][-1]),
    "learning_seconds": run["seconds"],
    f"V_at_{REF_PLAYER_SUM}_{REF_DEALER_CARD}": on_grid(run, budget, "values")[0, REPORT_STATE],
    "gap_at_reference": on_grid(run, budget, "values")[0, REPORT_STATE] - V_STAR[REPORT_STATE],
    "mean_gap_all_states": np.mean(on_grid(run, budget, "values")[0] - V_STAR),
    "loss_rate": on_grid(run, budget, "loss_rate")[0],
} for run in runs])
print(f"\nKL-SPMD + episodic TOMC, eta = {ETA_TUNED:g}, budget = {SAMPLE_BUDGET:,} observations")
print(summary.round(4).to_string(index=False))
print(f"\nGap at {REPORT_LABEL} (mean ± 95% CI): {format_mean_ci(summary['gap_at_reference'])}")
print(f"Mean gap over all states:        {format_mean_ci(summary['mean_gap_all_states'])}")
print(f"Loss rate (500 test games):      {format_mean_ci(summary['loss_rate'])}")
print(f"Policy updates per run:          {format_mean_ci(summary['policy_updates'])}")
print(f"Learning time per run:           {summary['learning_seconds'].mean():.0f} s")

## 11. Plots: value, loss rate, and policy updates

**Value.** Exact $V^{\pi_k}$ at the reference state, mean of five seeds, with its 95% band and
the optimal $V^*$. No running average: it would mix different policies. No policy can beat
$V^*$, so the value band is cut there.
**Loss rate.** Fraction of the 500 independent test games lost by the current policy.
**Policy updates.** Number of completed KL-SPMD updates, which depends on batch lengths.

In [ ]:
def apply_plot_style(large=False):
    plt.rcdefaults()
    plt.rcParams.update({
        "figure.facecolor": BACKGROUND, "axes.facecolor": BACKGROUND, "savefig.facecolor": BACKGROUND,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.spines.left": False, "axes.spines.bottom": False,
        "axes.grid": True, "grid.color": GRID_COLOR, "grid.linewidth": 0.8,
        "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlecolor": INK,
        "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
        "xtick.major.size": 0, "ytick.major.size": 0,
        "legend.frameon": False, "font.family": "DejaVu Sans",
        "font.size": 11, "axes.titlesize": 13, "axes.labelsize": 12, "legend.fontsize": 10.5,
    })
    if large:
        plt.rcParams.update({"font.size": 16, "axes.titlesize": 18, "axes.labelsize": 16,
                             "xtick.labelsize": 14, "ytick.labelsize": 14, "legend.fontsize": 14})


def plot_band(ax, points, curves, color, line_label, band_label=None, linewidth=2.0, lower=None):
    """Staircase mean across seeds with a shaded 95% confidence band.

    lower: a hard bound of the metric (V* for an exact value). The symmetric t-interval can
    cross it when seeds disagree (some converged, some not); the band is cut there.
    """
    mean, half_width = mean_ci(curves)
    low = mean - half_width if lower is None else np.maximum(mean - half_width, lower)
    ax.fill_between(points, low, mean + half_width, step="post",
                    color=color, alpha=0.2, linewidth=0, label=band_label)
    ax.step(points, mean, where="post", color=color, linewidth=linewidth, label=line_label)
    return mean, half_width


def save_figure(fig, name):
    FIGURES.mkdir(exist_ok=True)
    for suffix in ("png", "pdf"):
        fig.savefig(FIGURES / f"{name}.{suffix}", dpi=200, bbox_inches="tight")


millions = FuncFormatter(lambda x, _: f"{x / 1e6:g}M")
points = update_points(runs, SAMPLE_BUDGET)
value_curves = np.array([on_grid(run, points, "values")[:, REPORT_STATE] for run in runs])
loss_curves = np.array([on_grid(run, points, "loss_rate") for run in runs])
update_curves = np.array([on_grid(run, points, "updates") for run in runs])
v_star = V_STAR[REPORT_STATE]
n_seeds = len(runs)

# Overview: the three panels side by side (same layout as the GridWorld overview).
apply_plot_style()
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

ax = axes[0]
plot_band(ax, points, value_curves, BASELINE_COLOR,
          f"Exact $V^{{\\pi_k}}(s_0)$, mean of {n_seeds} seeds ± 95% CI", lower=v_star)
ax.axhline(v_star, color=INK, linewidth=1, linestyle="--", label="Optimal $V^*(s_0)$")
ax.set(title=f"Value at $s_0$ = {REPORT_LABEL}", ylabel="Discounted cost")

ax = axes[1]
plot_band(ax, points, loss_curves, BASELINE_COLOR,
          f"Loss rate, mean of {n_seeds} seeds ± 95% CI")
ax.set(title=f"Loss rate ({TEST_GAMES} independent test games)", ylabel="Loss rate")

ax = axes[2]
plot_band(ax, points, update_curves, BASELINE_COLOR,
          f"Completed updates, mean of {n_seeds} seeds ± 95% CI")
ax.set(title="Completed policy updates", ylabel="Policy updates")
ax.yaxis.set_major_locator(MaxNLocator(integer=True))

for ax in axes:
    ax.set(xlabel="Cumulative training observations", xlim=(0, SAMPLE_BUDGET))
    ax.xaxis.set_major_formatter(millions)
    ax.legend(loc="best")
fig.suptitle(f"Blackjack baseline: KL-SPMD + dynamic TOMC, $\\eta$ = {ETA_TUNED:g}",
             x=0.01, ha="left", fontsize=14, fontweight="bold")
fig.tight_layout()
save_figure(fig, "blackjack_baseline_results")
plt.show()

In [ ]:
# Large slide figures, one per metric (same size and fonts as the GridWorld slide figure).
def slide_figure(curves, title, ylabel, name, reference=None, reference_label=None,
                 value_format="{:.4f}", integer_axis=False, lower=None):
    apply_plot_style(large=True)
    fig, ax = plt.subplots(figsize=(11, 6.5))
    mean, half_width = plot_band(ax, points, curves, BASELINE_COLOR,
                                 f"Mean of {n_seeds} seeds", "95% confidence interval", linewidth=2.8,
                                 lower=lower)
    if reference is not None:
        ax.axhline(reference, color=INK, linewidth=1.8, linestyle="--", label=reference_label)
    end_text = f"{value_format.format(mean[-1])} ± {value_format.format(half_width[-1])}"
    ax.annotate(end_text, (points[-1], mean[-1]), xytext=(-10, 14), textcoords="offset points",
                ha="right", color=INK, fontsize=14)
    ax.annotate(f"uniform start {value_format.format(mean[0])}", (points[0], mean[0]),
                xytext=(10, 6), textcoords="offset points", color=MUTED, fontsize=13)
    ax.set(title=title, xlabel="Cumulative training observations", ylabel=ylabel,
           xlim=(0, SAMPLE_BUDGET))
    ax.xaxis.set_major_formatter(millions)
    if integer_axis:
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.legend(loc="best")
    fig.tight_layout()
    save_figure(fig, name)
    plt.show()


slide_figure(value_curves, "Blackjack baseline (KL-SPMD + TOMC)",
             f"Discounted cost at {REPORT_LABEL}", "blackjack_baseline_value",
             reference=v_star, reference_label=f"Optimal $V^*(s_0)$ = {v_star:.4f}", lower=v_star)
slide_figure(loss_curves, "Blackjack baseline: loss rate",
             f"Loss rate ({TEST_GAMES} test games)", "blackjack_baseline_loss_rate",
             value_format="{:.3f}")
slide_figure(update_curves, "Blackjack baseline: completed policy updates",
             "Cumulative policy updates", "blackjack_baseline_updates",
             value_format="{:.1f}", integer_axis=True)

## 12. Learned policy of the first seed

P(HIT) of the final policy next to the optimal action, for a quick check of what was learned.

In [ ]:
final = runs[0]["final_policy"]
policy_table = pd.DataFrame(
    [[f"{final[state2ID(p, d), HIT]:.2f} ({'SH'[OPTIMAL_ACTIONS[state2ID(p, d)]]})"
      for d in DEALER_CARDS] for p in PLAYER_SUMS],
    index=[f"sum {p}" for p in PLAYER_SUMS], columns=[f"dealer {d}" for d in DEALER_CARDS])
print(f"Seed {runs[0]['seed']}: final P(HIT), optimal action in brackets")
print(policy_table.to_string())